# Explainable Brain Tumor Detection - Comprehensive Demo

This notebook demonstrates the complete workflow for brain tumor detection and segmentation from MRI scans, with a focus on **explainability** and **robustness**.

## Workflow Overview:
1. **Data Loading & Visualization** - Load and explore MRI images
2. **Classification** - Predict tumor presence with confidence scores
3. **Explainability** - Generate Grad-CAM heatmaps to visualize model attention
4. **Uncertainty Estimation** - Quantify prediction confidence using MC Dropout
5. **Segmentation** - Predict tumor region boundaries (optional)
6. **Robustness Testing** - Test model behavior under image corruptions

## Setup & Imports

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Add src to path
ROOT_DIR = os.path.abspath('..')
sys.path.insert(0, ROOT_DIR)

# Core imports
import numpy as np
import torch
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path

# Project imports
from src.train import get_classifier, UNet
from src.preprocessing import (
    set_seed, list_image_files, get_segmentation_image, 
    get_segmentation_mask
)
from src.explainability import (
    GradCAM, overlay_heatmap, predict_with_uncertainty,
    load_image
)
from src.robustness import corrupt_image

# Setup
set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch version: {torch.__version__}')

## 1. Load Sample Images

Let's start by loading and visualizing sample MRI images from the dataset.

In [ ]:
# Find sample classification images
data_root = os.path.join(ROOT_DIR, 'data', 'classification', 'Testing')
image_files = []

if os.path.exists(data_root):
    for class_folder in os.listdir(data_root):
        class_path = os.path.join(data_root, class_folder)
        if os.path.isdir(class_path):
            images = list_image_files(class_path)
            if images:
                image_files.append((class_folder, images[0]))
                print(f'Found {len(images)} images in {class_folder}')
else:
    print(f'Classification data not found at {data_root}')
    print('Please ensure data is placed in data/classification/Testing/')

print(f'\nTotal classes found: {len(image_files)}')

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(1, min(3, len(image_files)), figsize=(12, 4))
if len(image_files) == 1:
    axes = [axes]

for idx, (class_name, image_path) in enumerate(image_files[:3]):
    img = Image.open(image_path).convert('RGB')
    axes[idx].imshow(img)
    axes[idx].set_title(f'{class_name}')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()
print(f'Displayed {min(3, len(image_files))} sample MRI images')

## 2. Load Pre-trained Models

Load the classification and segmentation models (if available).

In [ ]:
# Load classification model
classifier_path = os.path.join(ROOT_DIR, 'models', 'classifier.pth')
segmentation_path = os.path.join(ROOT_DIR, 'models', 'unet.pth')

classifier = None
segmentation_model = None

if os.path.exists(classifier_path):
    try:
        checkpoint = torch.load(classifier_path, map_location=device)
        classifier = get_classifier(num_classes=2, pretrained=False)
        classifier.load_state_dict(checkpoint['model_state_dict'])
        classifier = classifier.to(device).eval()
        print('✓ Classification model loaded successfully')
    except Exception as e:
        print(f'✗ Error loading classifier: {e}')
else:
    print(f'Classification model not found at {classifier_path}')
    print('Train a model first using: python src/train.py --task classification')

if os.path.exists(segmentation_path):
    try:
        checkpoint = torch.load(segmentation_path, map_location=device)
        segmentation_model = UNet(in_channels=3, out_channels=1)
        segmentation_model.load_state_dict(checkpoint['model_state_dict'])
        segmentation_model = segmentation_model.to(device).eval()
        print('✓ Segmentation model loaded successfully')
    except Exception as e:
        print(f'✗ Error loading segmentation model: {e}')
else:
    print(f'Segmentation model not found at {segmentation_path}')

## 3. Classification & Predictions

Make predictions on sample images.

In [ ]:
if classifier is None:
    print('Classifier not loaded. Skipping predictions.')
else:
    class_labels = ['No Tumor', 'Tumor']
    predictions = []
    
    for class_name, image_path in image_files:
        img = Image.open(image_path).convert('RGB')
        img_tensor = load_image(image_path, image_size=224).to(device)
        
        with torch.no_grad():
            output = classifier(img_tensor)
            scores = torch.softmax(output, dim=1).cpu().numpy()[0]
            pred_class = int(np.argmax(scores))
        
        predictions.append({
            'image': image_path,
            'true_class': class_name,
            'pred_class': class_labels[pred_class],
            'confidence': scores[pred_class],
            'scores': scores,
            'tensor': img_tensor
        })
        
        print(f'{class_name:15} → {class_labels[pred_class]:10} (conf: {scores[pred_class]:.3f})')
    
    print(f'\nProcessed {len(predictions)} images')

## 4. Explainability - Grad-CAM Heatmaps

Generate Grad-CAM heatmaps to visualize which regions of the image influence the model's predictions.

In [ ]:
if classifier is None or not predictions:
    print('Classifier or predictions not available. Skipping Grad-CAM.')
else:
    # Generate heatmaps for first 2 predictions
    fig, axes = plt.subplots(len(predictions[:2]), 2, figsize=(10, 5*len(predictions[:2])))
    if len(predictions[:2]) == 1:
        axes = axes.reshape(1, -1)
    
    target_layer = classifier.layer4[-1]
    gradcam = GradCAM(classifier, target_layer)
    
    for row_idx, pred_dict in enumerate(predictions[:2]):
        # Original image
        img = Image.open(pred_dict['image']).convert('RGB')
        img_224 = img.resize((224, 224))
        
        axes[row_idx, 0].imshow(img_224)
        axes[row_idx, 0].set_title(f'{pred_dict["pred_class"]} (conf: {pred_dict["confidence"]:.2%})')
        axes[row_idx, 0].axis('off')
        
        # Generate and overlay heatmap
        pred_class = int(np.argmax(pred_dict['scores']))
        heatmap = gradcam.generate(pred_dict['tensor'], class_idx=pred_class)
        overlay = overlay_heatmap(np.array(img_224), heatmap, alpha=0.6)
        
        axes[row_idx, 1].imshow(overlay)
        axes[row_idx, 1].set_title(f'Grad-CAM Attribution')
        axes[row_idx, 1].axis('off')
    
    plt.tight_layout()
    plt.show()
    print('Grad-CAM heatmaps generated')

## 5. Uncertainty Estimation

Use Monte Carlo Dropout to estimate prediction uncertainty and confidence.

In [ ]:
if classifier is None or not predictions:
    print('Classifier or predictions not available.')
else:
    print('Computing uncertainty estimates (MC Dropout)...')
    print('-' * 50)
    
    for pred_dict in predictions:
        uncertainty = predict_with_uncertainty(
            classifier, 
            pred_dict['tensor'].to(device),
            n_samples=10
        )
        
        pred_class_idx = uncertainty['predicted_class']
        pred_class_name = ['No Tumor', 'Tumor'][pred_class_idx]
        
        print(f'\nPrediction: {pred_class_name}')
        print(f'  Confidence: {uncertainty["mean_probability"][pred_class_idx]:.3f}')
        print(f'  Uncertainty: {uncertainty["uncertainty"]:.4f}')
        
        # Interpret uncertainty
        if uncertainty['uncertainty'] < 0.05:
            conf_level = 'High confidence'
        elif uncertainty['uncertainty'] < 0.15:
            conf_level = 'Medium confidence'
        else:
            conf_level = 'Low confidence (uncertain)'
        
        print(f'  → {conf_level}')

## 6. Robustness Testing

Test how the model responds to image corruptions (noise, blur, JPEG compression).

In [ ]:
if not predictions:
    print('No predictions available for robustness testing.')
else:
    # Test on first prediction
    pred_dict = predictions[0]
    img = Image.open(pred_dict['image']).convert('RGB')
    img_array = np.array(img.resize((224, 224))).astype(np.float32) / 255.0
    
    corruption_types = ['noise', 'blur', 'jpeg']
    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    
    # Original
    axes[0, 0].imshow(img.resize((224, 224)))
    axes[0, 0].set_title('Original')
    axes[0, 0].axis('off')
    
    # Corrupted versions
    for idx, corruption in enumerate(corruption_types):
        row = (idx + 1) // 2
        col = (idx + 1) % 2
        
        corrupted = corrupt_image(img_array, corruption, severity=0.1)
        axes[row, col].imshow(corrupted)
        axes[row, col].set_title(f'{corruption.capitalize()} (severity=0.1)')
        axes[row, col].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Test predictions on corrupted images
    if classifier is not None:
        print('\nPredictions on corrupted images:')
        print('-' * 50)
        
        for corruption in corruption_types:
            corrupted = corrupt_image(img_array, corruption, severity=0.1)
            corrupted_normalized = (
                corrupted - np.array([0.485, 0.456, 0.406])
            ) / np.array([0.229, 0.224, 0.225])
            corrupted_tensor = torch.from_numpy(
                corrupted_normalized.transpose(2, 0, 1)
            ).unsqueeze(0).float().to(device)
            
            with torch.no_grad():
                output = classifier(corrupted_tensor)
                scores = torch.softmax(output, dim=1).cpu().numpy()[0]
            
            pred_class = ['No Tumor', 'Tumor'][int(np.argmax(scores))]
            print(f'{corruption:10} → {pred_class:10} (conf: {scores[int(np.argmax(scores))]:.3f})')

## 7. Segmentation (Optional)

Generate tumor region segmentation masks if the segmentation model is available.

In [ ]:
# Load segmentation sample
seg_image_dir = os.path.join(ROOT_DIR, 'data', 'segmentation', 'images')
seg_mask_dir = os.path.join(ROOT_DIR, 'data', 'segmentation', 'masks')

seg_images = []
if os.path.exists(seg_image_dir) and os.path.exists(seg_mask_dir):
    images = sorted(list_image_files(seg_image_dir))[:2]
    
    for img_path in images:
        base = os.path.splitext(os.path.basename(img_path))[0]
        mask_path = os.path.join(seg_mask_dir, f'{base}_mask.tif')
        if os.path.exists(mask_path):
            seg_images.append((img_path, mask_path))
else:
    print('Segmentation data not found')

print(f'Found {len(seg_images)} segmentation image-mask pairs')

In [ ]:
if segmentation_model is None or not seg_images:
    print('Segmentation model or data not available.')
else:
    fig, axes = plt.subplots(len(seg_images), 3, figsize=(12, 4*len(seg_images)))
    if len(seg_images) == 1:
        axes = axes.reshape(1, -1)
    
    for row_idx, (img_path, mask_path) in enumerate(seg_images):
        # Load image and mask
        img = get_segmentation_image(img_path, image_size=(256, 256))
        true_mask = get_segmentation_mask(mask_path, image_size=(256, 256))
        
        # Predict mask
        img_tensor = torch.from_numpy(img.transpose(2, 0, 1)).unsqueeze(0).float().to(device)
        with torch.no_grad():
            pred_mask = torch.sigmoid(
                segmentation_model(img_tensor)
            ).cpu().numpy()[0, 0]
        
        # Visualize
        axes[row_idx, 0].imshow(img)
        axes[row_idx, 0].set_title('Input Image')
        axes[row_idx, 0].axis('off')
        
        axes[row_idx, 1].imshow(true_mask[..., 0], cmap='gray')
        axes[row_idx, 1].set_title('Ground Truth Mask')
        axes[row_idx, 1].axis('off')
        
        axes[row_idx, 2].imshow(pred_mask, cmap='gray')
        axes[row_idx, 2].set_title('Predicted Mask')
        axes[row_idx, 2].axis('off')
    
    plt.tight_layout()
    plt.show()
    print('Segmentation results displayed')

## Summary

This demo showcased the complete pipeline for **Explainable Brain Tumor Detection**:

✓ **Data Loading** - Loaded and visualized MRI images  
✓ **Classification** - Made predictions with confidence scores  
✓ **Explainability** - Generated Grad-CAM visualizations  
✓ **Uncertainty** - Estimated model confidence via MC Dropout  
✓ **Segmentation** - Predicted tumor region boundaries  
✓ **Robustness** - Tested resilience to image corruptions  

### Next Steps:
1. Train models with your own data using `src/train.py`
2. Evaluate performance with `src/evaluate.py`
3. Test robustness with `src/robustness.py`
4. Deploy with `streamlit run app/streamlit_app.py`

For more information, see `QUICKSTART.md` and `README.md`.